# 🎯 Module 3.2: Markov Decision Processes (MDPs)

## The Complete Mathematical Framework for Sequential Decision Making

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_03_RL_Mathematical_Foundations/02_Markov_Decision_Process/markov_decision_process.ipynb)

---

### 🎓 Learning Objectives

By the end of this notebook, you will be able to:

1. **Formally define all components of an MDP** — states, actions, transitions, rewards, and discount factor
2. **Understand state and action spaces for image processing** — and why they are astronomically large
3. **Define transition probabilities for deterministic image operations** — and contrast with stochastic transitions
4. **Analyze the role of the discount factor γ** — how it controls planning horizon and ensures convergence
5. **Build and visualize complete MDPs for image enhancement** — from grid worlds to real image processing

---

In [ ]:
!pip install numpy matplotlib networkx scipy pillow -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
np.random.seed(42)
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset for RL demonstrations
import torchvision
import numpy as np

# MNIST as discrete image states for MDP demonstrations
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True)
# Use small subset as "states" in our MDP
sample_states = [np.array(mnist[i][0]) for i in range(100)]
print(f"✅ MNIST loaded: Using {len(sample_states)} real images as MDP states")
print("   Each state is a 28x28 grayscale image = 784-dimensional state vector")

## Deep Derivation: MDP Formalism from First Principles

### Step 1: Formal MDP Definition
An MDP is a 5-tuple $\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, R, \gamma)$:
- $\mathcal{S}$: state space (finite or infinite)
- $\mathcal{A}$: action space (finite or infinite)
- $P: \mathcal{S} \times \mathcal{A} \times \mathcal{S} \to [0, 1]$: transition probability $P(s'|s, a)$
- $R: \mathcal{S} \times \mathcal{A} \times \mathcal{S} \to \mathbb{R}$: reward function
- $\gamma \in [0, 1)$: discount factor

### Step 2: Transition Probability — Proof of Stochastic Matrix Properties
For each $(s, a)$, the transition distribution must satisfy:
$$\sum_{s' \in \mathcal{S}} P(s'|s, a) = 1, \quad P(s'|s, a) \geq 0 \;\forall s'$$

**Proof that the transition matrix $\mathbf{P}_a$ is a stochastic matrix:**
Row $s$ of $\mathbf{P}_a$ is the probability distribution over next states given state $s$ and action $a$. By the axioms of probability, each row sums to 1 and all entries are non-negative. $\blacksquare$

**Proof that products of stochastic matrices are stochastic:**
If $\mathbf{P}$ and $\mathbf{Q}$ are stochastic, $(\mathbf{PQ})_{ij} = \sum_k P_{ik}Q_{kj} \geq 0$.
Row sum: $\sum_j (\mathbf{PQ})_{ij} = \sum_j \sum_k P_{ik}Q_{kj} = \sum_k P_{ik}\underbrace{\sum_j Q_{kj}}_{=1} = \sum_k P_{ik} = 1$ $\blacksquare$

This means the $n$-step transition probability $P^{(n)}(s'|s, a_1, \ldots, a_n)$ is also a valid distribution.

### Step 3: Discount Factor — Why $\gamma < 1$ is Necessary

**Theorem:** The discounted return $G_t = \sum_{k=0}^\infty \gamma^k R_{t+k+1}$ is bounded if rewards are bounded.

**Proof:** If $|R_t| \leq R_{\max}$ for all $t$:
$$|G_t| \leq \sum_{k=0}^\infty \gamma^k |R_{t+k+1}| \leq R_{\max} \sum_{k=0}^\infty \gamma^k = \frac{R_{\max}}{1 - \gamma} \quad \blacksquare$$

**For $\gamma = 1$ (undiscounted):** The sum may diverge! Consider $R_t = 1$ for all $t$: $G_t = \sum_{k=0}^\infty 1 = \infty$.

**Geometric series proof:** $\sum_{k=0}^{N} \gamma^k = \frac{1 - \gamma^{N+1}}{1 - \gamma}$

As $N \to \infty$ with $\gamma < 1$: $\gamma^{N+1} \to 0$, so $\sum_{k=0}^{\infty} \gamma^k = \frac{1}{1-\gamma}$ $\blacksquare$

### Step 4: State Space Size for Image MDPs

For a grayscale image of size $H \times W$ with $L$ intensity levels:
$$|\mathcal{S}| = L^{H \times W}$$

**Derivation:** Each of the $H \times W$ pixels independently takes one of $L$ values. By the multiplication principle, total combinations = $L^{H \times W}$.

| Image Size | Bit Depth | $|\mathcal{S}|$ | Feasible for Tabular RL? |
|-----------|-----------|-----------------|--------------------------|
| 4×4 | 2-bit | $4^{16} \approx 4.3 \times 10^9$ | Barely |
| 8×8 | 8-bit | $256^{64} \approx 10^{154}$ | No |
| 32×32 | 8-bit | $256^{1024} \approx 10^{2466}$ | Absolutely not |

### Step 5: Stationary Distribution (Ergodic MDPs)

For a fixed policy $\pi$, the Markov chain has transition matrix $\mathbf{P}^\pi$ where:
$$P^\pi(s'|s) = \sum_{a \in \mathcal{A}} \pi(a|s) P(s'|s, a)$$

**Theorem:** If the chain is ergodic (irreducible + aperiodic), a unique stationary distribution $\mathbf{d}^\pi$ exists:
$$\mathbf{d}^\pi = \mathbf{d}^\pi \mathbf{P}^\pi, \quad \sum_s d^\pi(s) = 1$$

**Proof sketch (Perron-Frobenius):** $\mathbf{P}^\pi$ is a non-negative irreducible matrix, so it has a unique largest eigenvalue $\lambda_1 = 1$ with positive eigenvector $\mathbf{d}^\pi$. $\blacksquare$

### RL Connection: Why MDPs are the Right Framework for Image Processing
Image enhancement is inherently sequential: apply filter → observe result → decide next operation. The MDP framework captures this because:
1. **State** = current image (Markov: only the current image matters, not how we got here)
2. **Action** = image processing operation to apply
3. **Transition** = deterministic: $I_{t+1} = f_a(I_t)$
4. **Reward** = image quality improvement (PSNR, SSIM change)

---

## 📖 Section 1: What is a Markov Decision Process?

A **Markov Decision Process (MDP)** is the mathematical framework that formalizes sequential decision-making under uncertainty. It provides the foundation for all of reinforcement learning.

### Formal Definition

An MDP is defined as a 5-tuple:

$$\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, R, \gamma)$$

Where:

| Component | Symbol | Description |
|-----------|--------|-------------|
| **State Space** | $\mathcal{S}$ | The set of all possible states (finite or infinite) |
| **Action Space** | $\mathcal{A}$ | The set of all possible actions (finite or infinite) |
| **Transition Function** | $P: \mathcal{S} \times \mathcal{A} \times \mathcal{S} \to [0,1]$ | Probability of reaching state $s'$ from state $s$ via action $a$: $P(s'|s,a)$ |
| **Reward Function** | $R: \mathcal{S} \times \mathcal{A} \to \mathbb{R}$ | Expected immediate reward for taking action $a$ in state $s$ |
| **Discount Factor** | $\gamma \in [0,1)$ | How much future rewards are worth compared to immediate rewards |

### The Markov Property

The key assumption is the **Markov property**:

$$P(s_{t+1} | s_t, a_t, s_{t-1}, a_{t-1}, \ldots, s_0, a_0) = P(s_{t+1} | s_t, a_t)$$

The future depends only on the **current state**, not the history of how we got there.

### Intuitive Explanation

Think of an MDP as a game:
- **States** = all possible situations you can be in
- **Actions** = all moves you can make
- **Transitions** = the rules of how the world changes when you act
- **Rewards** = points you earn/lose
- **Discount** = how impatient you are (prefer rewards now vs. later)

For **image processing**, the MDP looks like:
- **State** = the current image (or features of the image)
- **Action** = an image processing operation (brighten, sharpen, denoise, etc.)
- **Transition** = applying the operation transforms the image
- **Reward** = how much the image quality improved
- **Discount** = preference for fewer processing steps

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('The Agent-Environment Interaction in an MDP', fontsize=16, fontweight='bold', pad=20)

agent_box = mpatches.FancyBboxPatch((1, 3), 2.5, 2, boxstyle="round,pad=0.1",
                                     facecolor='#3498db', edgecolor='#2c3e50', linewidth=2)
ax.add_patch(agent_box)
ax.text(2.25, 4, 'AGENT', ha='center', va='center', fontsize=14, fontweight='bold', color='white')

env_box = mpatches.FancyBboxPatch((7.5, 3), 3, 2, boxstyle="round,pad=0.1",
                                   facecolor='#27ae60', edgecolor='#2c3e50', linewidth=2)
ax.add_patch(env_box)
ax.text(9, 4, 'ENVIRONMENT', ha='center', va='center', fontsize=14, fontweight='bold', color='white')

ax.annotate('', xy=(7.5, 4.7), xytext=(3.5, 4.7),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#e74c3c'))
ax.text(5.5, 5.2, 'Action $a_t$', ha='center', va='center', fontsize=12, color='#e74c3c', fontweight='bold')

ax.annotate('', xy=(3.5, 3.3), xytext=(7.5, 3.3),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#8e44ad'))
ax.text(5.5, 2.6, 'State $s_{t+1}$', ha='center', va='center', fontsize=12, color='#8e44ad', fontweight='bold')

ax.annotate('', xy=(3.5, 2.0), xytext=(7.5, 2.0),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#f39c12'))
ax.text(5.5, 1.4, 'Reward $r_{t+1}$', ha='center', va='center', fontsize=12, color='#f39c12', fontweight='bold')

components = [
    (1.0, 7.2, '$\\mathcal{S}$: State Space', '#3498db'),
    (4.0, 7.2, '$\\mathcal{A}$: Action Space', '#e74c3c'),
    (7.0, 7.2, '$P(s\'|s,a)$: Transitions', '#27ae60'),
    (10.0, 7.2, '$R(s,a)$: Rewards', '#f39c12'),
]

for x, y, text, color in components:
    ax.text(x, y, text, fontsize=11, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.1, edgecolor=color))

ax.text(6, 0.5, '$\\gamma \\in [0, 1)$: Discount Factor (controls planning horizon)',
        ha='center', fontsize=12, fontstyle='italic',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#ecf0f1', edgecolor='#7f8c8d'))

plt.tight_layout()
plt.show()
print("\n📊 The MDP framework: Agent observes state, takes action, receives reward, transitions to new state.")
print("   This cycle repeats, and the agent learns to maximize cumulative reward over time.")

## Deep Dive: Policy Evaluation as a Linear System

### The Bellman Expectation Equation

For a fixed policy $\pi$, the **value function** $V^\pi(s)$ satisfies the Bellman expectation equation:

$$V^\pi(s) = \sum_{a \in \mathcal{A}} \pi(a|s) \left[ R(s, a) + \gamma \sum_{s' \in \mathcal{S}} P(s'|s, a) \, V^\pi(s') \right]$$

**Step 1: Define the matrix form.** Let $|\mathcal{S}| = n$. Define:
- $\mathbf{V}^\pi \in \mathbb{R}^n$: the value vector with $V^\pi_i = V^\pi(s_i)$
- $\mathbf{R}^\pi \in \mathbb{R}^n$: the expected immediate reward vector, $R^\pi_i = \sum_a \pi(a|s_i) R(s_i, a)$
- $\mathbf{P}^\pi \in \mathbb{R}^{n \times n}$: the policy transition matrix, $P^\pi_{ij} = \sum_a \pi(a|s_i) P(s_j|s_i, a)$

The Bellman equation becomes the matrix equation:

$$\mathbf{V}^\pi = \mathbf{R}^\pi + \gamma \mathbf{P}^\pi \mathbf{V}^\pi$$

**Step 2: Rearrange to solve for $\mathbf{V}^\pi$:**

$$\mathbf{V}^\pi - \gamma \mathbf{P}^\pi \mathbf{V}^\pi = \mathbf{R}^\pi$$
$$(\mathbf{I} - \gamma \mathbf{P}^\pi) \mathbf{V}^\pi = \mathbf{R}^\pi$$

### Theorem: $(\mathbf{I} - \gamma \mathbf{P}^\pi)$ is Invertible When $\gamma < 1$

**Proof:** We show that all eigenvalues of $\gamma \mathbf{P}^\pi$ have magnitude strictly less than 1.

1. $\mathbf{P}^\pi$ is a **stochastic matrix** (rows sum to 1, all entries non-negative).

2. By the Gershgorin circle theorem, every eigenvalue $\lambda$ of $\mathbf{P}^\pi$ satisfies:
$$|\lambda - P^\pi_{ii}| \leq \sum_{j \neq i} P^\pi_{ij} = 1 - P^\pi_{ii}$$
   which gives $|\lambda| \leq 1$.

3. More precisely, for any stochastic matrix, $\|\mathbf{P}^\pi\|_\infty = \max_i \sum_j |P^\pi_{ij}| = 1$, so the spectral radius $\rho(\mathbf{P}^\pi) \leq 1$.

4. Therefore the eigenvalues of $\gamma \mathbf{P}^\pi$ satisfy:
$$|\gamma \lambda| = \gamma |\lambda| \leq \gamma < 1$$

5. Since all eigenvalues of $\gamma \mathbf{P}^\pi$ have magnitude $< 1$, the matrix $(\mathbf{I} - \gamma \mathbf{P}^\pi)$ has no zero eigenvalue, hence is **invertible**. $\blacksquare$

### The Closed-Form Solution

$$\boxed{\mathbf{V}^\pi = (\mathbf{I} - \gamma \mathbf{P}^\pi)^{-1} \mathbf{R}^\pi}$$

**Computational complexity:** Direct matrix inversion is $O(n^3)$, which is feasible for small state spaces but prohibitive for large MDPs (e.g., image states). This motivates iterative methods:
- **Iterative policy evaluation** (successive approximation): $\mathbf{V}_{k+1} = \mathbf{R}^\pi + \gamma \mathbf{P}^\pi \mathbf{V}_k$
- **Temporal-difference (TD) learning**: stochastic, sample-based updates
- **Function approximation**: neural networks to represent $V^\pi$

### Action-Value (Q) Function as a Linear System

Similarly, the action-value function $Q^\pi(s, a)$ satisfies:

$$Q^\pi(s, a) = R(s, a) + \gamma \sum_{s'} P(s'|s, a) \sum_{a'} \pi(a'|s') Q^\pi(s', a')$$

Defining $\mathbf{Q}^\pi \in \mathbb{R}^{n \cdot m}$ (for $m = |\mathcal{A}|$), this can be written as:

$$\mathbf{Q}^\pi = \mathbf{R} + \gamma \mathbf{P} \boldsymbol{\Pi} \mathbf{Q}^\pi$$

where $\boldsymbol{\Pi}$ is the block-diagonal policy matrix. The solution is $\mathbf{Q}^\pi = (\mathbf{I} - \gamma \mathbf{P}\boldsymbol{\Pi})^{-1}\mathbf{R}$.

**Connection to RL algorithms:** Value iteration and policy iteration are iterative procedures that avoid the expensive matrix inverse while converging to the same solution.

---

## 🖼️ Section 2: State Space for Images

### The Curse of Dimensionality

For a **grayscale image** of size $H \times W$ with $L$ intensity levels:

$$|\mathcal{S}| = L^{H \times W}$$

**Example**: An 8×8 grayscale image with 256 intensity levels:

$$|\mathcal{S}| = 256^{64} = 2^{512} \approx 1.34 \times 10^{154}$$

This is **astronomically large** — far more than the number of atoms in the observable universe ($\approx 10^{80}$)!

### Why This Matters

- We **cannot** enumerate all states
- We **cannot** store a table with one entry per state
- This motivates **function approximation** (neural networks)
- In practice, we use **compact representations**: features, embeddings, or learned encodings

### Practical State Representations

Instead of the raw pixel state, we can use:
- **Image statistics**: mean brightness, contrast, sharpness
- **Histogram features**: distribution of pixel values
- **Deep features**: CNN embeddings
- **Quality metrics**: PSNR, SSIM relative to target

In [ ]:
image_configs = [
    ('4x4', 4, 4, 256),
    ('8x8', 8, 8, 256),
    ('16x16', 16, 16, 256),
    ('32x32', 32, 32, 256),
    ('64x64', 64, 64, 256),
    ('128x128', 128, 128, 256),
    ('256x256', 256, 256, 256),
]

labels = []
log_state_sizes = []
actual_sizes = []

print("\ud83d\udcca State Space Size for Different Image Dimensions")
print("=" * 65)
print(f"{'Image Size':<12} {'Pixels':<10} {'|S| = 256^pixels':<25} {'Log10(|S|)'}")
print("-" * 65)

for name, h, w, levels in image_configs:
    pixels = h * w
    log10_size = pixels * np.log10(levels)
    labels.append(name)
    log_state_sizes.append(log10_size)
    print(f"{name:<12} {pixels:<10} 256^{pixels:<20} {log10_size:.1f}")

print("\n🌌 For reference:")
print("   Atoms in observable universe: ~10^80")
print("   Possible chess games: ~10^120")
print(f"   A tiny 8x8 image already has: ~10^{8*8*np.log10(256):.0f} states!")

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(labels)))
bars = ax.bar(labels, log_state_sizes, color=colors, edgecolor='black', linewidth=0.5)

ax.axhline(y=80, color='red', linestyle='--', linewidth=2, label='Atoms in universe (~$10^{80}$)')
ax.axhline(y=120, color='orange', linestyle='--', linewidth=2, label='Possible chess games (~$10^{120}$)')

ax.set_xlabel('Image Size', fontsize=12)
ax.set_ylabel('$\\log_{10}(|\\mathcal{S}|)$', fontsize=12)
ax.set_title('State Space Explosion for Images (256 Intensity Levels)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

for bar, val in zip(bars, log_state_sizes):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 500,
            f'$10^{{{int(val)}}}$', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Deep Dive: Neumann Series Expansion of the Value Function

### Motivation — Power Series for Matrices

From the closed-form $\mathbf{V}^\pi = (\mathbf{I} - \gamma \mathbf{P}^\pi)^{-1} \mathbf{R}^\pi$, we can expand the matrix inverse as a **Neumann series** (the matrix analog of the geometric series $\frac{1}{1-x} = \sum_{k=0}^\infty x^k$):

$$(\mathbf{I} - \gamma \mathbf{P}^\pi)^{-1} = \sum_{k=0}^{\infty} (\gamma \mathbf{P}^\pi)^k$$

### Theorem: Convergence of the Neumann Series

**Statement:** If $\rho(\mathbf{A}) < 1$ (spectral radius is less than 1), then $(\mathbf{I} - \mathbf{A})^{-1} = \sum_{k=0}^\infty \mathbf{A}^k$.

**Proof:**

1. Define the partial sum $\mathbf{S}_N = \sum_{k=0}^{N} \mathbf{A}^k$.

2. Observe the telescoping identity:
$$(\mathbf{I} - \mathbf{A}) \mathbf{S}_N = \mathbf{S}_N (\mathbf{I} - \mathbf{A}) = \mathbf{I} - \mathbf{A}^{N+1}$$

3. Since $\rho(\mathbf{A}) < 1$, we have $\|\mathbf{A}^{N+1}\| \to 0$ as $N \to \infty$ (every eigenvalue's magnitude is $< 1$, so $\mathbf{A}^N \to \mathbf{0}$).

4. Taking $N \to \infty$:
$$(\mathbf{I} - \mathbf{A}) \left(\sum_{k=0}^{\infty} \mathbf{A}^k\right) = \mathbf{I}$$
   
   Therefore $\sum_{k=0}^\infty \mathbf{A}^k = (\mathbf{I} - \mathbf{A})^{-1}$. $\blacksquare$

5. For $\mathbf{A} = \gamma \mathbf{P}^\pi$: we proved $\rho(\gamma \mathbf{P}^\pi) \leq \gamma < 1$, so the conditions are satisfied.

### The Value Function as an Infinite Horizon Sum

Substituting back:

$$\mathbf{V}^\pi = \sum_{k=0}^{\infty} (\gamma \mathbf{P}^\pi)^k \mathbf{R}^\pi = \mathbf{R}^\pi + \gamma \mathbf{P}^\pi \mathbf{R}^\pi + \gamma^2 (\mathbf{P}^\pi)^2 \mathbf{R}^\pi + \cdots$$

**Interpretation of each term:**

| Term | Expression | Meaning |
|------|-----------|---------|
| $k = 0$ | $\mathbf{R}^\pi$ | Immediate expected reward |
| $k = 1$ | $\gamma \mathbf{P}^\pi \mathbf{R}^\pi$ | Discounted reward one step ahead |
| $k = 2$ | $\gamma^2 (\mathbf{P}^\pi)^2 \mathbf{R}^\pi$ | Discounted reward two steps ahead |
| $k = n$ | $\gamma^n (\mathbf{P}^\pi)^n \mathbf{R}^\pi$ | Discounted reward $n$ steps ahead |

The entry $[(\mathbf{P}^\pi)^k]_{ij}$ is the probability of transitioning from state $s_i$ to state $s_j$ in exactly $k$ steps under policy $\pi$. So the value function sums up all future discounted rewards, weighted by their reachability.

### Convergence Rate

The truncation error after $N$ terms is:

$$\left\| \mathbf{V}^\pi - \sum_{k=0}^{N} (\gamma \mathbf{P}^\pi)^k \mathbf{R}^\pi \right\| \leq \frac{\gamma^{N+1}}{1 - \gamma} \|\mathbf{R}^\pi\|$$

**Derivation:** The remaining terms satisfy:
$$\left\|\sum_{k=N+1}^{\infty} \gamma^k (\mathbf{P}^\pi)^k \mathbf{R}^\pi\right\| \leq \sum_{k=N+1}^{\infty} \gamma^k \|\mathbf{R}^\pi\| = \frac{\gamma^{N+1}}{1-\gamma}\|\mathbf{R}^\pi\|$$

using $\|(\mathbf{P}^\pi)^k\|_\infty \leq 1$ (product of stochastic matrices is stochastic). $\blacksquare$

**Practical implication:** With $\gamma = 0.99$ and $R_{\max} = 1$, to achieve $\epsilon = 0.01$ accuracy:
$$\frac{0.99^{N+1}}{0.01} \leq 0.01 \implies N \geq \frac{\log(10^{-4})}{\log(0.99)} \approx 917 \text{ steps}$$

This quantifies the "effective horizon" of the MDP — beyond $\sim \frac{1}{1-\gamma}$ steps, future rewards are negligible.

### Connection to Iterative Policy Evaluation

The iterative update $\mathbf{V}_{k+1} = \mathbf{R}^\pi + \gamma \mathbf{P}^\pi \mathbf{V}_k$ starting from $\mathbf{V}_0 = \mathbf{0}$ produces exactly the partial sums of the Neumann series:
$$\mathbf{V}_k = \sum_{j=0}^{k-1} (\gamma \mathbf{P}^\pi)^j \mathbf{R}^\pi$$

This proves that **iterative policy evaluation converges** to the true value function, with geometric convergence rate $\gamma$.

In [ ]:
def compute_brightness(img):
    """Mean pixel intensity."""
    return np.mean(img)

def compute_contrast(img):
    """Standard deviation of pixel intensities."""
    return np.std(img)

def compute_sharpness(img):
    """Laplacian variance as sharpness measure."""
    laplacian = ndimage.laplace(img.astype(float))
    return np.var(laplacian)

np.random.seed(42)
n_images = 30
images = []
features = []
categories = []

for i in range(10):
    img = np.random.randint(150, 255, (32, 32)).astype(np.uint8)
    img = ndimage.gaussian_filter(img, sigma=1)
    images.append(img)
    categories.append('Bright')

for i in range(10):
    img = np.random.randint(0, 100, (32, 32)).astype(np.uint8)
    img = ndimage.gaussian_filter(img, sigma=2)
    images.append(img)
    categories.append('Dark')

for i in range(10):
    img = np.random.randint(0, 255, (32, 32)).astype(np.uint8)
    images.append(img)
    categories.append('High Contrast')

for img in images:
    b = compute_brightness(img)
    c = compute_contrast(img)
    s = compute_sharpness(img)
    features.append([b, c, s])

features = np.array(features)

fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(121, projection='3d')
color_map = {'Bright': '#f39c12', 'Dark': '#2c3e50', 'High Contrast': '#e74c3c'}
for cat in ['Bright', 'Dark', 'High Contrast']:
    mask = [c == cat for c in categories]
    ax1.scatter(features[mask, 0], features[mask, 1], features[mask, 2],
               label=cat, c=color_map[cat], s=60, alpha=0.8, edgecolors='black', linewidth=0.5)

ax1.set_xlabel('Brightness', fontsize=10)
ax1.set_ylabel('Contrast', fontsize=10)
ax1.set_zlabel('Sharpness', fontsize=10)
ax1.set_title('3D State Space\n(Feature Representation)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)

ax2 = fig.add_subplot(122)
sample_indices = [0, 10, 20]
for idx, i in enumerate(sample_indices):
    ax_img = fig.add_axes([0.58 + idx * 0.14, 0.55, 0.12, 0.3])
    ax_img.imshow(images[i], cmap='gray', vmin=0, vmax=255)
    ax_img.set_title(f'{categories[i]}\nB={features[i,0]:.0f}, C={features[i,1]:.0f}', fontsize=8)
    ax_img.axis('off')

ax2.axis('off')
ax2.text(0.5, 0.25, 'Compact state representation:\n\n'
         'Raw image: 32×32 = 1024 dimensions\n'
         'Feature state: 3 dimensions\n\n'
         'Reduction: 99.7% fewer dimensions!',
         ha='center', va='center', fontsize=11,
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#ecf0f1', edgecolor='#34495e'))

plt.tight_layout()
plt.show()



## Deep Dive: State-Action Occupancy Measure

### Definition

The **discounted state-action occupancy measure** (or visitation frequency) under policy $\pi$ starting from initial distribution $\rho_0$ is:

$$d^\pi(s, a) = (1 - \gamma) \sum_{t=0}^{\infty} \gamma^t \Pr(s_t = s, a_t = a \mid \pi, \rho_0)$$

The marginal **state occupancy** is $d^\pi(s) = \sum_a d^\pi(s, a)$, and the conditional satisfies $d^\pi(s, a) = d^\pi(s) \pi(a|s)$.

The normalizing factor $(1 - \gamma)$ ensures $\sum_{s,a} d^\pi(s,a) = 1$, making it a proper probability distribution.

### Derivation: Connecting Occupancy to Performance

**Step 1:** Write the performance objective in terms of occupancy:

$$J(\pi) = \mathbb{E}_\pi\left[\sum_{t=0}^\infty \gamma^t r_t\right] = \sum_{t=0}^\infty \gamma^t \sum_{s,a} \Pr(s_t=s, a_t=a|\pi) R(s,a)$$

$$= \frac{1}{1-\gamma} \sum_{s,a} d^\pi(s,a) R(s,a) = \frac{1}{1-\gamma} \mathbb{E}_{(s,a) \sim d^\pi}[R(s,a)]$$

**Step 2:** This means the expected return is just the expected reward under the occupancy measure (up to scaling).

### Theorem: Bijection Between Policies and Occupancy Measures

**Statement (Puterman, 1994):** For any valid occupancy measure $d(s,a) \geq 0$ satisfying the **Bellman flow constraints**:

$$\sum_a d(s', a) = (1 - \gamma) \rho_0(s') + \gamma \sum_{s,a} d(s, a) P(s'|s, a) \quad \forall s'$$

there exists a unique policy $\pi_d$ that generates it, given by:

$$\pi_d(a|s) = \frac{d(s, a)}{\sum_{a'} d(s, a')}$$

**Proof sketch:**

1. **Forward direction** ($\pi \to d^\pi$): Given policy $\pi$, the occupancy measure $d^\pi$ is uniquely determined by unrolling the Markov chain: $d^\pi(s,a) = (1-\gamma)\sum_{t=0}^\infty \gamma^t P(s_t=s|\pi)\pi(a|s)$.

2. **Backward direction** ($d \to \pi_d$): Given valid $d$, define $\pi_d(a|s) = d(s,a)/\sum_{a'} d(s,a')$. Verify this policy reproduces $d$ by substituting into the Bellman flow equation. $\blacksquare$

### Why This Matters: Linear Programming Formulation of MDPs

This bijection allows us to reformulate the MDP as a **linear program**:

$$\max_{d \geq 0} \sum_{s,a} d(s,a) R(s,a)$$
$$\text{subject to: } \sum_a d(s', a) = (1-\gamma)\rho_0(s') + \gamma \sum_{s,a} d(s,a) P(s'|s,a) \quad \forall s'$$

This is a convex optimization problem with $|\mathcal{S}| \times |\mathcal{A}|$ variables and $|\mathcal{S}|$ equality constraints. It can be solved in polynomial time, providing an alternative to dynamic programming.

### The Policy Gradient Through Occupancy

The policy gradient theorem can be re-derived using occupancy measures:

$$\nabla_\theta J(\theta) = \frac{1}{1-\gamma} \mathbb{E}_{(s,a) \sim d^{\pi_\theta}} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot Q^{\pi_\theta}(s,a) \right]$$

This formulation makes explicit that the gradient is an expectation over the discounted state-action distribution — not just over single trajectories. It is the foundation of modern policy gradient algorithms.

## Deep Dive: Discount Factor — Probabilistic Interpretation and Average Reward

### Two Interpretations of $\gamma$

The discount factor $\gamma \in [0, 1)$ admits two fundamentally different but mathematically equivalent interpretations:

---

### Interpretation 1: Geometric Discounting of Future Rewards

The most common view — future rewards are worth less than immediate ones:

$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$$

The weight on a reward $k$ steps in the future is $\gamma^k$. After $n = \frac{1}{1-\gamma}$ steps, the cumulative weight exceeds $1 - e^{-1} \approx 63\%$ of the total.

| $\gamma$ | Effective horizon $\frac{1}{1-\gamma}$ | Character |
|----------|----------------------------------------|-----------|
| 0.0 | 1 step | Completely myopic |
| 0.9 | 10 steps | Short-sighted |
| 0.99 | 100 steps | Moderate planning |
| 0.999 | 1000 steps | Far-sighted |
| 1.0 | $\infty$ | Undiscounted (may diverge) |

---

### Interpretation 2: Probability of Episode Continuation

**Theorem:** Discounted returns are equivalent to undiscounted returns in an MDP where the episode terminates at each step with probability $(1 - \gamma)$.

**Proof:** Define a modified MDP $\tilde{\mathcal{M}}$ identical to $\mathcal{M}$ except that at each time step, with probability $(1 - \gamma)$, the episode terminates (transitions to an absorbing state with zero reward).

The probability the episode survives to step $k$ is $\gamma^k$. The expected undiscounted return in $\tilde{\mathcal{M}}$:

$$\tilde{G}_t = \mathbb{E}\left[\sum_{k=0}^{\tilde{T}} r_{t+k+1}\right]$$

where $\tilde{T} \sim \text{Geometric}(1 - \gamma)$.

Computing:
$$\mathbb{E}[\tilde{G}_t] = \sum_{k=0}^{\infty} \Pr(\tilde{T} \geq k) \cdot \mathbb{E}[r_{t+k+1}] = \sum_{k=0}^{\infty} \gamma^k \cdot \mathbb{E}[r_{t+k+1}] = \mathbb{E}[G_t] \quad \blacksquare$$

**Intuition:** Discounting is not about "preferring now over later" but about modeling an uncertain, finite lifetime.

---

### The Average Reward Formulation (When $\gamma \to 1$)

For ergodic MDPs, an alternative to discounting is the **average reward** criterion:

$$\bar{r}^\pi = \lim_{T \to \infty} \frac{1}{T} \mathbb{E}_\pi\left[\sum_{t=0}^{T-1} r_t\right] = \sum_s d^\pi(s) \sum_a \pi(a|s) R(s, a)$$

where $d^\pi(s)$ is the stationary distribution of the Markov chain under $\pi$.

**Theorem (Laurent series connection):** As $\gamma \to 1$:

$$(1 - \gamma) V^\pi_\gamma(s) \to \bar{r}^\pi \quad \forall s$$

**Proof sketch:** Using the Neumann series:
$$(1-\gamma) V^\pi_\gamma(s) = (1-\gamma) \sum_{k=0}^{\infty} \gamma^k [\mathbf{P}^k \mathbf{R}^\pi]_s$$

As $\gamma \to 1$, this becomes a Cesàro average of $[(\mathbf{P}^\pi)^k \mathbf{R}^\pi]_s$. By the ergodic theorem, $(\mathbf{P}^\pi)^k \to \mathbf{1} (\mathbf{d}^\pi)^T$, so:

$$(1-\gamma) V^\pi_\gamma(s) \to (\mathbf{d}^\pi)^T \mathbf{R}^\pi = \bar{r}^\pi \quad \blacksquare$$

**Practical note:** Most deep RL algorithms use discounting ($\gamma = 0.99$) rather than average reward because: (a) it guarantees convergence of the return, (b) it induces a well-defined contraction mapping for value iteration, and (c) it naturally handles episodic tasks with terminal states.

---

### The Discount Factor in Image Processing MDPs

For image enhancement, $\gamma$ controls how many processing steps the agent considers:
- **Low $\gamma$** ($\approx 0.5$): Agent applies one or two filters greedily
- **High $\gamma$** ($\approx 0.99$): Agent plans long sequences of complementary operations
- **Sweet spot** ($\gamma \approx 0.9$–$0.95$): Balances quick improvements with multi-step planning

## Deep Dive: Bellman Optimality and the Contraction Mapping Theorem

### The Bellman Optimality Equations

The **optimal value function** $V^*(s) = \max_\pi V^\pi(s)$ satisfies:

$$V^*(s) = \max_{a \in \mathcal{A}} \left[ R(s, a) + \gamma \sum_{s'} P(s'|s, a) \, V^*(s') \right]$$

Similarly, the **optimal action-value function** $Q^*(s, a) = \max_\pi Q^\pi(s, a)$:

$$Q^*(s, a) = R(s, a) + \gamma \sum_{s'} P(s'|s, a) \max_{a'} Q^*(s', a')$$

The optimal policy is the greedy policy with respect to $Q^*$:

$$\pi^*(a|s) = \begin{cases} 1 & \text{if } a = \arg\max_{a'} Q^*(s, a') \\ 0 & \text{otherwise} \end{cases}$$

### The Bellman Operator

Define the **Bellman optimality operator** $\mathcal{T}^*: \mathbb{R}^{|\mathcal{S}|} \to \mathbb{R}^{|\mathcal{S}|}$:

$$[\mathcal{T}^* V](s) = \max_{a} \left[ R(s, a) + \gamma \sum_{s'} P(s'|s, a) \, V(s') \right]$$

The optimal value function is the **fixed point**: $\mathcal{T}^* V^* = V^*$.

### Theorem: $\mathcal{T}^*$ is a $\gamma$-Contraction in the Supremum Norm

**Statement:** For any $V_1, V_2 \in \mathbb{R}^{|\mathcal{S}|}$:

$$\|\mathcal{T}^* V_1 - \mathcal{T}^* V_2\|_\infty \leq \gamma \|V_1 - V_2\|_\infty$$

**Proof:**

For any state $s$:
$$[\mathcal{T}^* V_1](s) = \max_a \left[R(s,a) + \gamma \sum_{s'} P(s'|s,a) V_1(s')\right]$$

Let $a_1^* = \arg\max_a [R(s,a) + \gamma \sum_{s'} P(s'|s,a) V_1(s')]$.

Then:
$$[\mathcal{T}^* V_1](s) - [\mathcal{T}^* V_2](s) \leq R(s, a_1^*) + \gamma \sum_{s'} P(s'|s,a_1^*) V_1(s') - R(s, a_1^*) - \gamma \sum_{s'} P(s'|s,a_1^*) V_2(s')$$

$$= \gamma \sum_{s'} P(s'|s,a_1^*) [V_1(s') - V_2(s')]$$

$$\leq \gamma \sum_{s'} P(s'|s,a_1^*) \|V_1 - V_2\|_\infty = \gamma \|V_1 - V_2\|_\infty$$

By symmetry (swapping $V_1$ and $V_2$), $[\mathcal{T}^* V_2](s) - [\mathcal{T}^* V_1](s) \leq \gamma \|V_1 - V_2\|_\infty$.

Therefore $|[\mathcal{T}^* V_1](s) - [\mathcal{T}^* V_2](s)| \leq \gamma \|V_1 - V_2\|_\infty$ for all $s$, giving:

$$\|\mathcal{T}^* V_1 - \mathcal{T}^* V_2\|_\infty \leq \gamma \|V_1 - V_2\|_\infty \quad \blacksquare$$

### Corollary: Convergence of Value Iteration

By the **Banach fixed-point theorem**, since $\mathcal{T}^*$ is a $\gamma$-contraction on the complete metric space $(\mathbb{R}^{|\mathcal{S}|}, \|\cdot\|_\infty)$:

1. A **unique** fixed point $V^*$ exists.
2. The iteration $V_{k+1} = \mathcal{T}^* V_k$ converges to $V^*$ from **any** initial $V_0$.
3. The convergence rate is **geometric**: $\|V_k - V^*\|_\infty \leq \gamma^k \|V_0 - V^*\|_\infty$.

**Number of iterations for $\epsilon$-optimality:**

$$k \geq \frac{\log(\|V_0 - V^*\|_\infty / \epsilon)}{\log(1/\gamma)} = \frac{\log(\|V_0 - V^*\|_\infty / \epsilon)}{-\log \gamma} \approx \frac{1}{1-\gamma} \log\frac{\|V_0 - V^*\|_\infty}{\epsilon}$$

This is one of the most elegant results in RL theory: the discount factor $\gamma$ simultaneously determines the planning horizon AND the convergence rate of the fundamental optimization algorithm.

## Deep Dive: Stationary Distribution — Existence, Uniqueness, and Convergence

### Setup: The Policy-Induced Markov Chain

For a fixed policy $\pi$, the MDP reduces to a Markov chain with transition matrix $\mathbf{P}^\pi$:

$$P^\pi(s'|s) = \sum_{a \in \mathcal{A}} \pi(a|s) P(s'|s, a)$$

### Theorem (Perron-Frobenius): Existence of Stationary Distribution

**Statement:** If $\mathbf{P}^\pi$ is the transition matrix of an **irreducible** (all states communicate) and **aperiodic** (GCD of return times is 1) Markov chain, then:

1. There exists a **unique** stationary distribution $\mathbf{d}^\pi \in \mathbb{R}^{|\mathcal{S}|}$ satisfying $(\mathbf{d}^\pi)^T \mathbf{P}^\pi = (\mathbf{d}^\pi)^T$
2. $d^\pi(s) > 0$ for all $s \in \mathcal{S}$
3. $(\mathbf{P}^\pi)^n \to \mathbf{1}(\mathbf{d}^\pi)^T$ as $n \to \infty$ (convergence to stationarity)

**Proof outline:**

**Step 1 — Existence via eigenvalue analysis:**

$\mathbf{P}^\pi$ is a non-negative matrix with row sums equal to 1. Therefore $\mathbf{1}^T \mathbf{P}^\pi = \mathbf{1}^T$ does not directly help, but $(\mathbf{P}^\pi)^T$ has $\lambda = 1$ as an eigenvalue (since $(\mathbf{P}^\pi)^T \mathbf{1} = \mathbf{1}$, but we need the left eigenvector of $\mathbf{P}^\pi$).

By the Perron-Frobenius theorem for non-negative irreducible matrices, $\mathbf{P}^\pi$ has a unique largest eigenvalue $\lambda_1 = 1$, and the corresponding left eigenvector $\mathbf{d}^\pi$ can be chosen to have all positive entries and to satisfy $\sum_s d^\pi(s) = 1$.

**Step 2 — Uniqueness:**

Suppose $\mathbf{d}_1$ and $\mathbf{d}_2$ are both stationary distributions. Define $\mathbf{w} = \mathbf{d}_1 - \mathbf{d}_2$, so $\mathbf{w}^T \mathbf{P}^\pi = \mathbf{w}^T$ and $\mathbf{w}^T \mathbf{1} = 0$.

By irreducibility, for any $i, j$, there exists $n$ such that $[(\mathbf{P}^\pi)^n]_{ij} > 0$. This forces all entries of $\mathbf{w}$ to have the same sign (otherwise we could find a contradiction with $\mathbf{w}^T \mathbf{1} = 0$). Combined with $\mathbf{w}^T \mathbf{1} = 0$, this implies $\mathbf{w} = \mathbf{0}$, so $\mathbf{d}_1 = \mathbf{d}_2$. $\blacksquare$

**Step 3 — Convergence rate:**

The rate of convergence to stationarity is governed by the **spectral gap** $1 - |\lambda_2|$, where $\lambda_2$ is the second-largest eigenvalue of $\mathbf{P}^\pi$:

$$\|(\mathbf{P}^\pi)^n_{s,\cdot} - \mathbf{d}^\pi\|_{TV} \leq C \cdot |\lambda_2|^n$$

The **mixing time** (time to reach approximate stationarity) is $t_{\text{mix}} \approx \frac{1}{1 - |\lambda_2|}$.

### Practical Computation of $\mathbf{d}^\pi$

For small state spaces, solve the eigenvalue problem directly:

$$(\mathbf{d}^\pi)^T (\mathbf{P}^\pi - \mathbf{I}) = \mathbf{0}, \quad \sum_s d^\pi(s) = 1$$

This is a system of $|\mathcal{S}|$ linear equations (one is redundant, replaced by the normalization constraint).

For large state spaces, use **power iteration**: start with any distribution $\mathbf{d}_0$ and compute $\mathbf{d}_{k+1}^T = \mathbf{d}_k^T \mathbf{P}^\pi$ until convergence.

### Significance for RL

The stationary distribution determines:
1. **Which states the agent visits most** — critical for understanding exploration
2. **The average reward**: $\bar{r}^\pi = \sum_s d^\pi(s) R^\pi(s)$
3. **The variance of gradient estimates** in on-policy methods — states with low $d^\pi(s)$ contribute rare, high-variance samples
4. **State-action occupancy**: $d^\pi(s, a) \propto d^\pi(s) \pi(a|s)$, which connects to the linear programming formulation of MDPs